# 48 â€” TabNet / Attention-Based Tabular Learning

Attention-based tabular model (TabNet if available, else custom AttentionMLP fallback).
Especially suited for sparse feature interactions in Morgan FP + RDKit descriptor space.

**Primary metric:** RAE (lower is better). Current best OOF RAE: 0.5281.

In [1]:
import os as _os
_torch_lib = r"d:\Users\ashenoy00000\.windsurf\OpenADMET-pxr-challenge\.venv\Lib\site-packages\torch\lib"
if _os.path.exists(_torch_lib):
    _os.add_dll_directory(_torch_lib)

import sys, os
os.environ["PYTHONIOENCODING"] = "utf-8"
sys.path.insert(0, "../src")
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import RidgeCV
import lightgbm as lgb
from pxr.data import load_train, load_test
from pxr.featurize import combined, impute
from pxr.eval import rae, scaffold_kfold_indices
from pxr.chem import bemis_murcko, morgan_fp_batch
from pxr.paths import DATA_PROCESSED, SUBMISSIONS

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
LGBM_PARAMS = dict(n_estimators=1000, num_leaves=64, learning_rate=0.05,
                   subsample=0.8, colsample_bytree=0.8, reg_alpha=0.1,
                   reg_lambda=0.1, min_child_samples=10, n_jobs=4, verbose=-1)
print(f"Device: {DEVICE}")

# Check for pytorch_tabnet
try:
    from pytorch_tabnet.tab_model import TabNetRegressor
    USE_TABNET = True
    print("pytorch_tabnet available â€” will use TabNetRegressor")
except ImportError:
    USE_TABNET = False
    print("pytorch_tabnet not available â€” will use AttentionMLP fallback")

Device: cpu
pytorch_tabnet available â€” will use TabNetRegressor


## 1. Load data

In [2]:
tr = load_train()
te = load_test()
print(f"Train: {len(tr)} | Test: {len(te)}")

tr['scaffold'] = tr['smiles'].apply(bemis_murcko)
y_tr = tr['pec50'].values.astype(np.float32)

X_tr_raw = combined(tr['smiles'].tolist())
X_te_raw = combined(te['smiles'].tolist())
X_tr_raw = impute(X_tr_raw)
X_te_raw = impute(X_te_raw)
print(f"X_tr: {X_tr_raw.shape} | X_te: {X_te_raw.shape}")

Train: 4139 | Test: 513


X_tr: (4139, 2265) | X_te: (513, 2265)


## 2. Model definitions

In [3]:
class AttentionMLP(nn.Module):
    """Simplified TabNet-like multi-step attention mechanism."""
    def __init__(self, d_in=2265, n_steps=3, d_attn=128, d_out=1):
        super().__init__()
        self.d_in = d_in
        self.n_steps = n_steps
        self.steps = nn.ModuleList([
            nn.Sequential(
                nn.Linear(d_in, d_attn * 2),
                nn.GLU(dim=-1),  # Gated linear: halves dim back to d_attn
            )
            for _ in range(n_steps)
        ])
        # Attention over input features
        self.attn_weights = nn.ModuleList([
            nn.Sequential(
                nn.Linear(d_in, d_in),
                nn.Softmax(dim=-1)
            )
            for _ in range(n_steps)
        ])
        self.bn = nn.ModuleList([nn.BatchNorm1d(d_attn) for _ in range(n_steps)])
        self.output = nn.Linear(n_steps * d_attn, d_out)

    def forward(self, x):
        h_steps = []
        for step_net, attn_net, bn in zip(self.steps, self.attn_weights, self.bn):
            weights = attn_net(x)          # (B, d_in) â€” soft feature selection
            masked_x = x * weights
            h = step_net(masked_x)         # (B, d_attn) after GLU
            h = bn(h)
            h_steps.append(h)
        out = self.output(torch.cat(h_steps, dim=-1))
        return out.squeeze(-1)

    def get_attention_weights(self, x):
        """Return attention weights from each step for feature importance."""
        all_weights = []
        for attn_net in self.attn_weights:
            all_weights.append(attn_net(x).detach())
        return torch.stack(all_weights, dim=1)  # (B, n_steps, d_in)


print("AttentionMLP defined.")
d_in = X_tr_raw.shape[1]
test_model = AttentionMLP(d_in=d_in, n_steps=3, d_attn=128)
n_params = sum(p.numel() for p in test_model.parameters())
print(f"Parameters: {n_params:,}")

AttentionMLP defined.
Parameters: 17,138,911


## 3. Training utilities

In [4]:
def train_attention_mlp(
    X_train, y_train,
    n_steps=3, d_attn=128,
    n_epochs=150, batch_size=256, lr=1e-3,
    device=DEVICE,
):
    d_in = X_train.shape[1]
    scaler = StandardScaler()
    X_sc = scaler.fit_transform(X_train).astype(np.float32)

    model = AttentionMLP(d_in=d_in, n_steps=n_steps, d_attn=d_attn).to(device)
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=n_epochs)
    loss_fn = nn.HuberLoss(delta=1.0)

    X_t = torch.tensor(X_sc, dtype=torch.float32)
    y_t = torch.tensor(y_train, dtype=torch.float32)
    ds = TensorDataset(X_t, y_t)
    loader = DataLoader(ds, batch_size=batch_size, shuffle=True, drop_last=False)

    best_loss = float('inf')
    best_state = None
    model.train()
    for epoch in range(n_epochs):
        epoch_loss = 0.0
        for x_b, y_b in loader:
            x_b, y_b = x_b.to(device), y_b.to(device)
            optimizer.zero_grad()
            pred = model(x_b)
            loss = loss_fn(pred, y_b)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            epoch_loss += loss.item()
        scheduler.step()
        if epoch_loss < best_loss:
            best_loss = epoch_loss
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
        if (epoch + 1) % 50 == 0:
            print(f"    Epoch {epoch+1}/{n_epochs}  loss={epoch_loss:.4f}")

    if best_state is not None:
        model.load_state_dict(best_state)
    return model, scaler


@torch.no_grad()
def predict_attention_mlp(model, scaler, X, batch_size=512, device=DEVICE):
    model.eval()
    X_sc = scaler.transform(X).astype(np.float32)
    X_t = torch.tensor(X_sc, dtype=torch.float32)
    preds = []
    for i in range(0, len(X_t), batch_size):
        preds.append(model(X_t[i:i+batch_size].to(device)).cpu().numpy())
    return np.concatenate(preds)


def train_tabnet(X_train, y_train, X_val=None, y_val=None):
    """Train using pytorch_tabnet TabNetRegressor."""
    scaler = StandardScaler()
    X_sc = scaler.fit_transform(X_train).astype(np.float32)

    eval_set = None
    if X_val is not None and y_val is not None:
        X_val_sc = scaler.transform(X_val).astype(np.float32)
        eval_set = [(X_val_sc, y_val.reshape(-1, 1))]

    model = TabNetRegressor(
        n_d=64, n_a=64, n_steps=5, gamma=1.5,
        lambda_sparse=1e-4, optimizer_fn=optim.AdamW,
        optimizer_params={'lr': 2e-3, 'weight_decay': 1e-4},
        scheduler_fn=optim.lr_scheduler.CosineAnnealingLR,
        scheduler_params={'T_max': 200},
        verbose=0, device_name='cuda' if torch.cuda.is_available() else 'cpu',
    )
    model.fit(
        X_sc, y_train.reshape(-1, 1),
        eval_set=eval_set,
        eval_metric=['mae'],
        max_epochs=200,
        patience=20,
        batch_size=256,
    )
    return model, scaler


def predict_tabnet(model, scaler, X):
    X_sc = scaler.transform(X).astype(np.float32)
    return model.predict(X_sc).flatten()


print("Training utilities defined.")

Training utilities defined.


## 4. Scaffold 5-fold CV

In [5]:
splits = scaffold_kfold_indices(tr['scaffold'], n_splits=5)
oof_preds = np.full(len(tr), np.nan)
fold_raes = []
fold_models = []

for fold, (tr_idx, val_idx) in enumerate(splits):
    print(f"\nFold {fold+1}/5 â€” train={len(tr_idx)}, val={len(val_idx)}")

    X_fold_tr = X_tr_raw[tr_idx]
    X_fold_val = X_tr_raw[val_idx]
    y_fold_tr = y_tr[tr_idx]
    y_fold_val = y_tr[val_idx]

    if USE_TABNET:
        model, scaler = train_tabnet(X_fold_tr, y_fold_tr, X_fold_val, y_fold_val)
        val_preds = predict_tabnet(model, scaler, X_fold_val)
    else:
        model, scaler = train_attention_mlp(
            X_fold_tr, y_fold_tr, n_steps=3, d_attn=128,
            n_epochs=150, batch_size=256, lr=1e-3,
        )
        val_preds = predict_attention_mlp(model, scaler, X_fold_val)

    fold_rae = rae(y_fold_val, val_preds)
    fold_raes.append(fold_rae)
    oof_preds[val_idx] = val_preds
    fold_models.append((model, scaler))
    print(f"  Fold RAE: {fold_rae:.4f}")

oof_rae = rae(y_tr, oof_preds)
print(f"\n=== OOF RAE: {oof_rae:.4f} (mean fold: {np.mean(fold_raes):.4f} Â± {np.std(fold_raes):.4f}) ===")


Fold 1/5 â€” train=3311, val=828



Early stopping occurred at epoch 116 with best_epoch = 96 and best_val_0_mae = 0.61481


D:\Users\ashenoy00000\.windsurf\OpenADMET-pxr-challenge\.venv\Lib\site-packages\pytorch_tabnet\callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)


  Fold RAE: 0.6110

Fold 2/5 â€” train=3311, val=828



Early stopping occurred at epoch 91 with best_epoch = 71 and best_val_0_mae = 0.74514


...

D:\Users\ashenoy00000\.windsurf\OpenADMET-pxr-challenge\.venv\Lib\site-packages\pytorch_tabnet\callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)


  Fold RAE: 0.8721

Fold 5/5 â€” train=3312, val=827



Early stopping occurred at epoch 98 with best_epoch = 78 and best_val_0_mae = 0.73285


D:\Users\ashenoy00000\.windsurf\OpenADMET-pxr-challenge\.venv\Lib\site-packages\pytorch_tabnet\callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)


  Fold RAE: 0.8375

=== OOF RAE: 0.7904 (mean fold: 0.7998 Â± 0.0954) ===


## 5. Feature importance via attention weights

In [6]:
from pxr.featurize import feature_names_rdkit

rdkit_feat_names = feature_names_rdkit()
n_morgan = 2048
feature_names = [f'morgan_{i}' for i in range(n_morgan)] + rdkit_feat_names
# Trim to actual feature count
feature_names = feature_names[:X_tr_raw.shape[1]]

if not USE_TABNET and len(fold_models) > 0:
    # Collect attention weights from last fold's model
    last_model, last_scaler = fold_models[-1]
    last_model.eval()

    # Subsample training data for efficiency
    rng = np.random.default_rng(42)
    sample_idx = rng.choice(len(X_tr_raw), size=min(500, len(X_tr_raw)), replace=False)
    X_sample = last_scaler.transform(X_tr_raw[sample_idx]).astype(np.float32)
    X_sample_t = torch.tensor(X_sample, dtype=torch.float32).to(DEVICE)

    with torch.no_grad():
        attn_weights = last_model.get_attention_weights(X_sample_t)  # (B, n_steps, d_in)
    mean_attn = attn_weights.mean(dim=(0, 1)).cpu().numpy()  # (d_in,)

    top_k = 50
    top_idx = np.argsort(mean_attn)[::-1][:top_k]
    print(f"Top-{top_k} most attended features:")
    morgan_count = sum(1 for i in top_idx if i < n_morgan)
    rdkit_count = top_k - morgan_count
    print(f"  Morgan FP bits: {morgan_count} / {top_k}")
    print(f"  RDKit descriptors: {rdkit_count} / {top_k}")
    top_feat_df = pd.DataFrame({
        'feature': [feature_names[i] for i in top_idx],
        'mean_attention': mean_attn[top_idx],
        'type': ['morgan' if i < n_morgan else 'rdkit' for i in top_idx],
    })
    print(top_feat_df.head(20).to_string(index=False))
elif USE_TABNET and len(fold_models) > 0:
    last_model, last_scaler = fold_models[-1]
    # TabNet provides feature importances
    importances = last_model.feature_importances_
    top_idx = np.argsort(importances)[::-1][:50]
    morgan_count = sum(1 for i in top_idx if i < n_morgan)
    print(f"TabNet top-50 features: {morgan_count} Morgan, {50-morgan_count} RDKit")
else:
    print("No models available for feature importance.")

TabNet top-50 features: 42 Morgan, 8 RDKit


## 6. Final model â€” train on all data, predict test

In [7]:
print("Training final model on all training data...")
if USE_TABNET:
    final_model, final_scaler = train_tabnet(X_tr_raw, y_tr)
    test_preds = predict_tabnet(final_model, final_scaler, X_te_raw)
else:
    final_model, final_scaler = train_attention_mlp(
        X_tr_raw, y_tr, n_steps=3, d_attn=128,
        n_epochs=150, batch_size=256, lr=1e-3,
    )
    test_preds = predict_attention_mlp(final_model, final_scaler, X_te_raw)

y_lo = y_tr.min() - 0.5
y_hi = y_tr.max() + 0.5
test_preds = np.clip(test_preds, y_lo, y_hi)
print(f"Test pred range: [{test_preds.min():.3f}, {test_preds.max():.3f}]")

Training final model on all training data...


D:\Users\ashenoy00000\.windsurf\OpenADMET-pxr-challenge\.venv\Lib\site-packages\pytorch_tabnet\abstract_model.py:687: UserWarning: No early stopping will be performed, last training weights will be used.
  warnings.warn(wrn_msg)


Test pred range: [2.717, 6.765]


In [8]:
# Save OOF
np.save(DATA_PROCESSED / 'oof_tabnet.npy', oof_preds)
print("Saved OOF predictions.")

# Save submission
sub = pd.DataFrame({'Molecule Name': te['name'], 'pEC50': test_preds})
out_path = SUBMISSIONS / '48_tabnet_pxr.csv'
sub.to_csv(out_path, index=False)
print(f"Saved submission to {out_path}")
print(sub.head())
print(f"\nFinal OOF RAE: {oof_rae:.4f}")

Saved OOF predictions.
Saved submission to D:\Users\ashenoy00000\.windsurf\OpenADMET-pxr-challenge\submissions\48_tabnet_pxr.csv
    Molecule Name     pEC50
0  OADMET-0006617  4.603793
1  OADMET-0006616  3.624328
2  OADMET-0006615  5.112897
3  OADMET-0006614  4.390078
4  OADMET-0006613  4.984620

Final OOF RAE: 0.7904
